# 02 - Geocoding

This notebook converts employee addresses to geographic coordinates using the LocationIQ API. We'll add caching to avoid making repeated API calls for the same addresses, which saves time and respects API rate limits.

In [ ]:
# Import the libraries we need for geocoding
import requests
import time
import pandas as pd
from dotenv import load_dotenv
import os
import json

In [ ]:
# Load environment variables from .env file to get our API keys
load_dotenv()

# Get LocationIQ API key for geocoding
LOCATIONIQ_API_KEY = os.getenv('LOCATIONIQ_API_KEY')

if not LOCATIONIQ_API_KEY:
    raise ValueError("LOCATIONIQ_API_KEY not found in environment variables")

# Get OpenRouteService API key (we'll use this later for routing)
ORS_API_KEY = os.getenv('OPENROUTESERVICE_API_KEY')

In [ ]:
# Function to geocode a single address using LocationIQ API
def geocode_with_locationiq(address, api_key):
    """Geocode a single address using LocationIQ"""
    base_url = "https://us1.locationiq.com/v1/search.php"
    
    params = {
        'key': api_key,
        'q': address,
        'format': 'json',
        'limit': 1
    }
    
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
        
        if data and len(data) > 0:
            lat = float(data[0]['lat'])
            lon = float(data[0]['lon'])
            return lat, lon
        else:
            return None, None
            
    except Exception as e:
        print(f"Error geocoding {address}: {e}")
        return None, None

In [ ]:
# Geocoding cache file to store results and avoid repeated API calls
GEOCODING_CACHE_FILE = 'data/geocoding_cache.csv'

def load_geocoding_cache():
    """Load existing geocoding cache if it exists"""
    if os.path.exists(GEOCODING_CACHE_FILE):
        try:
            cache_df = pd.read_csv(GEOCODING_CACHE_FILE)
            return dict(zip(cache_df['address'], zip(cache_df['latitude'], cache_df['longitude'])))
        except Exception as e:
            print(f"Error loading cache: {e}")
            return {}
    return {}

def save_geocoding_cache(cache):
    """Save geocoding cache to file so we don't lose progress"""
    try:
        cache_df = pd.DataFrame([
            {'address': addr, 'latitude': lat, 'longitude': lon}
            for addr, (lat, lon) in cache.items()
        ])
        cache_df.to_csv(GEOCODING_CACHE_FILE, index=False)
        print(f"Cache saved with {len(cache)} entries")
    except Exception as e:
        print(f"Error saving cache: {e}")

def geocode_employee_addresses(df, api_key):
    """Geocode all employee addresses with caching to save time and API quota"""
    cache = load_geocoding_cache()
    coordinates = []
    cache_hits = 0
    cache_misses = 0
    
    for idx, row in df.iterrows():
        address = row['address']
        
        # Check cache first - if we already geocoded this address, use the cached result
        if address in cache:
            lat, lon = cache[address]
            coordinates.append((lat, lon))
            cache_hits += 1
            print(f"Cache hit {idx+1}/{len(df)}: {address}")
        else:
            print(f"Geocoding {idx+1}/{len(df)}: {address}")
            lat, lon = geocode_with_locationiq(address, api_key)
            coordinates.append((lat, lon))
            cache_misses += 1
            
            # Add to cache
            cache[address] = (lat, lon)
            
            # Save cache periodically so we don't lose progress if something fails
            if (idx + 1) % 10 == 0:
                save_geocoding_cache(cache)
            
            # Rate limiting to respect API limits
            time.sleep(0.5)
    
    df['latitude'] = [coord[0] for coord in coordinates]
    df['longitude'] = [coord[1] for coord in coordinates]
    
    # Save final cache
    save_geocoding_cache(cache)
    
    print(f"\nGeocoding complete: {cache_hits} cache hits, {cache_misses} new API calls")
    
    return df

In [ ]:
# Load our synthetic employee data
employee_df = pd.read_csv('data/synthetic_employees.csv')

# Geocode all the addresses (this will use caching if we've already done this before)
employee_df = geocode_employee_addresses(employee_df, LOCATIONIQ_API_KEY)

In [ ]:
# Check our geocoding results
print(employee_df[['employee_id', 'address', 'latitude', 'longitude']].head())
print(f"Successfully geocoded {employee_df['latitude'].notna().sum()}/{len(employee_df)} addresses")

In [ ]:
# Function to snap coordinates to nearest walkable street using OpenRouteService API
# This is important for realistic routing - we want coordinates on actual streets, not random points


def snap_coordinates_to_street(lat, lon, api_key, radius=500):
    """Snap coordinates to nearest walkable street using OpenRouteService API"""
    url = "https://api.openrouteservice.org/v2/snap/foot-walking"

    headers = {
        "Authorization": api_key,
        "Content-Type": "application/json"
    }

    body = {
        "locations": [[lon, lat]],
        "radius": radius  # Increased radius to find more street options
    }

    # Try up to 3 times in case of rate limiting
    for attempt in range(3):
        response = requests.post(
            url,
            json=body,
            headers=headers
        )

        if response.status_code == 200:
            data = response.json()
            snapped = data["locations"][0]["location"]
            return snapped[1], snapped[0]  # Return lat, lon

        elif response.status_code == 429:
            print("Rate limited. Waiting...")
            time.sleep(5)

        else:
            print(response.text)
            break

    return lat, lon  # Return original coordinates if snapping fails

In [ ]:
# Remove any addresses that failed to geocode
employee_df_clean = employee_df.dropna(subset=['latitude', 'longitude'])
print(f"Original: {len(employee_df)} employees")
print(f"After removing failed geocodes: {len(employee_df_clean)} employees")
print(f"Success rate: {len(employee_df_clean)/len(employee_df)*100:.1f}%")

In [ ]:
# First, let's snap the J&J office coordinates to nearest street for realistic routing
# We'll snap employee coordinates in the next cell
jj_address = 'Robert-Koch-Straße 1, 22851 Norderstedt'
jj_lat, jj_lon = 53.6876, 10.0053  # Correct J&J office coordinates

print(f"J&J Office address: {jj_address}")
print(f"Original J&J office coordinates: ({jj_lat}, {jj_lon})")

# Snap the office coordinates to nearest street
snapped_jj_lat, snapped_jj_lon = snap_coordinates_to_street(jj_lat, jj_lon, ORS_API_KEY)

print(f"Snapped J&J office coordinates: ({snapped_jj_lat}, {snapped_jj_lon})")

# Save snapped office coordinates to JSON file for use in other notebooks
jj_office_data = {
    'address': jj_address,
    'original_latitude': jj_lat,
    'original_longitude': jj_lon,
    'snapped_latitude': snapped_jj_lat,
    'snapped_longitude': snapped_jj_lon
}

with open('data/jj_office_snapped.json', 'w') as f:
    json.dump(jj_office_data, f, indent=2)

print("Saved snapped J&J office coordinates to data/jj_office_snapped.json")

In [ ]:
# Snap all employee coordinates to nearest streets for realistic routing
# This takes a while due to API rate limiting, but it's important for accurate routing results
snapped_coords = []
for idx, row in employee_df_clean.iterrows():
    lat, lon = row['latitude'], row['longitude']
    snapped_lat, snapped_lon = snap_coordinates_to_street(lat, lon, ORS_API_KEY)
    snapped_coords.append((snapped_lat, snapped_lon))
    print(f"Snapped {idx+1}/{len(employee_df_clean)}: ({lat:.4f}, {lon:.4f}) -> ({snapped_lat:.4f}, {snapped_lon:.4f})")
    time.sleep(0.1)  # Rate limiting to avoid API limits

employee_df_clean['snapped_latitude'] = [coord[0] for coord in snapped_coords]
employee_df_clean['snapped_longitude'] = [coord[1] for coord in snapped_coords]

print(f"Successfully snapped {len(employee_df_clean)} coordinates")

In [ ]:
# Check our final results including both original and snapped coordinates
print(employee_df_clean[['employee_id', 'address', 'latitude', 'longitude', 'snapped_latitude', 'snapped_longitude']].head())
print(f"Successfully geocoded {employee_df_clean['latitude'].notna().sum()}/{len(employee_df_clean)} addresses")
print(f"Successfully snapped {employee_df_clean['snapped_latitude'].notna().sum()}/{len(employee_df_clean)} coordinates")

In [ ]:
# Check if any coordinates failed to snap (some addresses might not have nearby walkable streets)
failed_snaps = employee_df_clean[employee_df_clean['snapped_latitude'] == employee_df_clean['latitude']]
print(f"Coordinates that couldn't be snapped: {len(failed_snaps)}")
if len(failed_snaps) > 0:
    print("These coordinates will use original geocoded values")

# Save our final geocoded data with snapped coordinates for use in routing notebooks
employee_df_clean.to_csv('data/synthetic_employees_geocoded.csv', index=False)
print("Saved geocoded employee data with snapped coordinates to data/synthetic_employees_geocoded.csv")